# SmolVLA Fine-Tuning & Evaluation on LIBERO-Spatial
**robotics project Assessment** | Joshua Wisdom Momo | CMU MS AI

Pipeline: Setup → Train (20K steps) → Eval Fine-tuned → Eval Reference → Backup

**Hardware**: T4 16GB (training), A100 80GB (evaluation)
**Key overrides**: batch_size=2 (T4 OOM at 8), eval_freq=0 (eval envs consume 5.6GB VRAM)

In [10]:
# GPU verification + Drive mount
import torch, os

assert torch.cuda.is_available(), "No GPU -- switch runtime to GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/smolvla_project'
for subdir in ['checkpoints', 'videos', 'results', 'figures']:
    os.makedirs(f'{DRIVE_BASE}/{subdir}', exist_ok=True)
print(f"Drive backup dir: {DRIVE_BASE}")

GPU: NVIDIA A100-SXM4-80GB (79.3 GB)
PyTorch: 2.10.0+cu128 | CUDA: 12.8
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive backup dir: /content/drive/MyDrive/smolvla_project


In [11]:
# Install dependencies (~5 min)
!pip install -q "lerobot[libero] @ git+https://github.com/huggingface/lerobot.git"
!pip install -q num2words matplotlib seaborn

# Fix egl_probe cmake version
!git clone -q https://github.com/StanfordVL/egl_probe.git /tmp/egl_probe 2>/dev/null || true
!cd /tmp/egl_probe && sed -i 's/cmake_minimum_required(VERSION 2.8.12)/cmake_minimum_required(VERSION 3.10)/' egl_probe/CMakeLists.txt && pip install -q . 2>/dev/null

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done


In [12]:
# Set environment variables and verify
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import lerobot, mujoco
print(f"LeRobot: {getattr(lerobot, '__version__', 'installed')}")
print(f"MuJoCo: {mujoco.__version__}")
print(f"MUJOCO_GL={os.environ['MUJOCO_GL']}")

LeRobot: 0.5.1
MuJoCo: 3.6.0
MUJOCO_GL=egl


In [13]:
# Verify environment
try:
    import lerobot
    print(f"LeRobot version: {getattr(lerobot, '__version__', 'installed')}")
except Exception as e:
    print(f"LeRobot import issue: {e}")

try:
    import mujoco
    print(f"MuJoCo version: {mujoco.__version__}")
except Exception as e:
    print(f"MuJoCo import issue: {e}")

print("\nEnvironment ready.")


LeRobot version: 0.5.1
MuJoCo version: 3.6.0

Environment ready.


In [ ]:
# Cell 4: Wandb login
!wandb login

# Use Wandb Report magic link in SUBMISSION.md instead.

## Part 2: Baseline Training (20K steps, chunk_size=50)
Training the Action Expert (approx. 100M params) while VLM backbone (approx. 350M) stays frozen.
Expected: approx. 2h 48min on T4, approx 45min on A100.

In [ ]:
# Baseline training
import time
start_time = time.time()
print(f"Training started at: {time.strftime('%H:%M:%S')}")

!lerobot-train \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.push_to_hub=false \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --batch_size=2 \
  --steps=20000 \
  --eval_freq=0 \
  --save_freq=5000 \
  --policy.use_amp=true \
  --wandb.enable=true \
  --wandb.project=smolvla-libero \
  --output_dir=outputs/train/smolvla_libero_spatial \
  --seed=42

elapsed = (time.time() - start_time) / 60
print(f"\nTraining completed in {elapsed:.1f} minutes")

In [ ]:
# Verify checkpoints, backup to Drive, save metadata
import os, shutil, json, time as _time

ckpt_dir = 'outputs/train/smolvla_libero_spatial/checkpoints'
dst = f'{DRIVE_BASE}/checkpoints/baseline'

# Verify
print("Checkpoints found:")
for d in sorted(os.listdir(ckpt_dir)):
    full = os.path.join(ckpt_dir, d)
    if os.path.isdir(full):
        size_mb = sum(os.path.getsize(os.path.join(dp, f))
                      for dp, _, fns in os.walk(full) for f in fns) / (1024**2)
        print(f"  {d}/  ({size_mb:.0f} MB)")

# Backup to Drive
t0 = _time.time()
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(ckpt_dir, dst)
print(f"\nBacked up to Drive in {_time.time()-t0:.0f}s")

# Save training metadata
training_meta = {
    "run_name": "baseline_chunk50_20k",
    "model": "lerobot/smolvla_base",
    "dataset": "HuggingFaceVLA/libero",
    "task": "libero_spatial",
    "steps": 20000,
    "batch_size": 2,
    "seed": 42,
    "eval_freq": 0,
    "save_freq": 5000,
    "use_amp": True,
    "chunk_size": 50,
    "num_denoising_steps": 10,
    "gpu": gpu_name,
    "gpu_memory_gb": round(gpu_mem, 1),
    "training_time_minutes": round(elapsed, 1),
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "config_overrides": {
        "batch_size": "2 (reduced from default 8 to fit T4 16GB)",
        "eval_freq": "0 (eval envs consume 5.6GB VRAM at startup)",
    },
    "notes": "Baseline run. VLM backbone frozen, only Action Expert (~100M params) trains.",
}
meta_path = f'{DRIVE_BASE}/training_meta_baseline.json'
with open(meta_path, 'w') as f:
    json.dump(training_meta, f, indent=2)
print(f"Saved metadata: {meta_path}")


## Part 3: Evaluation
Evaluate fine-tuned (20K checkpoint) and reference (HuggingFaceVLA/smolvla_libero) models.
10 tasks x 10 episodes = 100 episodes per model.

**Important**: Use batch_size=10 on A100. Output redirected to Drive to prevent page hanging.

In [ ]:
# Restore checkpoints from Drive (run after runtime restart)
!mkdir -p /content/outputs/train/smolvla_libero_spatial/
!cp -r /content/drive/MyDrive/smolvla_project/checkpoints/baseline \
       /content/outputs/train/smolvla_libero_spatial/checkpoints
!ls /content/outputs/train/smolvla_libero_spatial/checkpoints/020000/pretrained_model/config.json

In [ ]:
# Evaluate fine-tuned model (output to Drive only)
!echo "N" | lerobot-eval \
  --policy.path=/content/outputs/train/smolvla_libero_spatial/checkpoints/020000/pretrained_model \
  --env.type=libero \
  --env.task=libero_spatial \
  --eval.batch_size=10 \
  --eval.n_episodes=10 \
  --eval.use_async_envs=false \
  --policy.use_amp=true \
  --output_dir=/content/outputs/eval/smolvla_finetuned \
  --seed=42 \
  > /content/drive/MyDrive/smolvla_project/eval_finetuned_log.txt 2>&1

print("FINETUNED EVAL DONE")

In [ ]:
# Backup fine-tuned eval to Drive
!cp -r /content/outputs/eval/smolvla_finetuned /content/drive/MyDrive/smolvla_project/eval_finetuned
print("Fine-tuned eval backed up to Drive")

In [ ]:
# Evaluate reference model (output to Drive only)
!echo "N" | lerobot-eval \
  --policy.path=HuggingFaceVLA/smolvla_libero \
  --env.type=libero \
  --env.task=libero_spatial \
  --eval.batch_size=10 \
  --eval.n_episodes=10 \
  --eval.use_async_envs=false \
  --policy.use_amp=true \
  --output_dir=/content/outputs/eval/smolvla_reference \
  --seed=42 \
  > /content/drive/MyDrive/smolvla_project/eval_reference_log.txt 2>&1

print("REFERENCE EVAL DONE")

In [ ]:
# Backup reference eval to Drive
!cp -r /content/outputs/eval/smolvla_reference /content/drive/MyDrive/smolvla_project/eval_reference
print("Reference eval backed up to Drive")

In [ ]:
# Cell 12: Print results summary
import json, glob

for label, path in [("FINETUNED", "/content/outputs/eval/smolvla_finetuned"),
                    ("REFERENCE", "/content/outputs/eval/smolvla_reference")]:
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    for f in glob.glob(f"{path}/**/*.json", recursive=True):
        try:
            data = json.load(open(f))
            print(json.dumps(data, indent=2)[:800])
        except:
            pass
    vids = glob.glob(f"{path}/**/*.mp4", recursive=True)
    print(f"  Videos: {len(vids)}")

## Troubleshooting

**Training OOM on T4**: Reduce batch_size to 1, verify eval_freq=0.

**Training interrupted**: Resume with:
```
lerobot-train --resume=true \
  --config_path=outputs/train/.../checkpoints/last/pretrained_model/train_config.json \
  --eval_freq=0
```

**Eval silent crash**: System RAM OOM. Use High-RAM runtime or A100.

**Eval path error** (`HFValidationError`): Use absolute paths: `/content/outputs/...`

**Page Unresponsive during eval**: Redirect output to Drive file, not notebook.

**libero config path error**: `!rm -f ~/.libero/config.yaml`